# Project: Code Review Crew

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) CrewAI roadmap.

You will build a hierarchical crew: a manager coordinates security, performance, and best-practices reviewers, then synthesizes a structured `CodeReview` with optional human input.

## 1. Install dependencies

In [ ]:
!pip install -q crewai crewai-tools pydantic

## 2. Set your API key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Step 1 — Custom tool (`@tool`)

A tool that takes a code string and returns it, simulating a file read.

In [ ]:
from crewai.tools import tool

@tool("Read Code Snippet")
def read_code_snippet(code: str) -> str:
    """Simulate reading source from a file. Pass the full source code as the code argument."""
    return code

## 4. Step 2 — Pydantic models (`CodeReview`, `Issue`)

In [ ]:
from pydantic import BaseModel

class Issue(BaseModel):
    severity: str
    description: str
    suggestion: str

class CodeReview(BaseModel):
    file_name: str
    issues: list[Issue]
    overall_score: int
    summary: str

## 5. Step 3 — Three specialist agents

In [ ]:
from crewai import Agent

security_reviewer = Agent(
    role="Security Reviewer",
    goal="Find security vulnerabilities, unsafe patterns, and auth/data-handling risks",
    backstory="You specialize in secure coding, OWASP-style thinking, and threat-aware review.",
    tools=[read_code_snippet],
    verbose=True,
    allow_delegation=False,
)

performance_reviewer = Agent(
    role="Performance Reviewer",
    goal="Identify performance bottlenecks, unnecessary work, and scalability concerns",
    backstory="You profile and reason about complexity, I/O, and hot paths in application code.",
    tools=[read_code_snippet],
    verbose=True,
    allow_delegation=False,
)

best_practices_reviewer = Agent(
    role="Best Practices Reviewer",
    goal="Assess readability, maintainability, naming, structure, and idiomatic style",
    backstory="You care about clean code, consistency, and long-term maintainability.",
    tools=[read_code_snippet],
    verbose=True,
    allow_delegation=False,
)

## 6. Step 4 — Review tasks and synthesis (`output_pydantic`, `human_input`)

In [ ]:
from crewai import Task

security_review = Task(
    description=(
        "Review the code for file {file_name}. Use read_code_snippet with code={code} to load the snippet, "
        "then report security issues, unsafe assumptions, and hardening suggestions."
    ),
    expected_output="Concise security-focused findings the manager can merge into a final review.",
)

performance_review = Task(
    description=(
        "Review the code for file {file_name}. Use read_code_snippet with code={code}, "
        "then report performance risks, complexity concerns, and concrete optimizations."
    ),
    expected_output="Concise performance-focused findings the manager can merge into a final review.",
)

practices_review = Task(
    description=(
        "Review the code for file {file_name}. Use read_code_snippet with code={code}, "
        "then report style, structure, naming, and maintainability improvements."
    ),
    expected_output="Concise best-practices findings the manager can merge into a final review.",
)

synthesis_task = Task(
    description=(
        "Combine the security, performance, and best-practices findings for {file_name} into one review. "
        "Merge overlapping points, deduplicate, and assign severities. "
        "Set overall_score from 1 (poor) to 10 (excellent). "
        "Populate CodeReview: file_name, issues (severity, description, suggestion), overall_score, summary."
    ),
    expected_output="A single validated CodeReview object.",
    context=[security_review, performance_review, practices_review],
    output_pydantic=CodeReview,
    human_input=True,
)

## 7. Step 5 — Crew with `Process.hierarchical` and `manager_llm`

In [ ]:
from crewai import Crew, Process

crew = Crew(
    agents=[security_reviewer, performance_reviewer, best_practices_reviewer],
    tasks=[security_review, performance_review, practices_review, synthesis_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o",
    verbose=True,
)

## 8. Step 6 — `kickoff` and read `result.pydantic`

When prompted for human input, type feedback in the notebook input area (or approve as needed). Then inspect the structured review.

In [ ]:
sample_code = """
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + str(user_id)
    return db.execute(query)
"""

result = crew.kickoff(inputs={"file_name": "users.py", "code": sample_code})

review = result.pydantic
print(review.file_name, review.overall_score)
print(review.summary)
for issue in review.issues:
    print(issue.severity, issue.description, "->", issue.suggestion)

## What you learned

- **Hierarchical crews** — `Process.hierarchical` with `manager_llm` coordinates specialist agents without assigning each task to an agent manually.
- **Custom tools** — `@tool` wraps a function so reviewers call `read_code_snippet` like a file read.
- **Structured output** — `output_pydantic=CodeReview` validates the final artifact; use `result.pydantic` for typed fields.
- **Human in the loop** — `human_input=True` on synthesis pauses for your feedback before the final structured `CodeReview` is fixed.